In [1]:
import random

import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cpu")

## Question 1 — loss and output contract

Which loss matches the model above?

- `nn.MSELoss()`
- `nn.BCEWithLogitsLoss()`
- `nn.CrossEntropyLoss()`
- `nn.CosineEmbeddingLoss()`

There is one intended answer: the model emits one logit and the loss applies the
stable sigmoid-plus-binary-cross-entropy operation.



## Question 2 — parameter count

What is the total number of trainable parameters? Count biases as well and use
`sum(p.numel() for p in model.parameters())`.

- `896`
- `11214912`
- `15896912`
- `20073473`


In [2]:
import torch.nn as nn


class HairModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=0)
        self.pool = nn.MaxPool2d(kernel_size=2)
        self.hidden = nn.Linear(32 * 99 * 99, 64)
        self.output = nn.Linear(64, 1)

    def forward(self, x):
        x = self.pool(nn.functional.relu(self.conv(x)))
        x = x.flatten(start_dim=1)
        x = nn.functional.relu(self.hidden(x))
        return self.output(x)

In [3]:
model = HairModel()
print(sum(p.numel() for p in model.parameters()))

20073473



## Training metrics

Train for 10 epochs with the baseline transform. For every epoch, evaluate the
held-out split with `model.eval()` and `torch.no_grad()`. Store these four
series in a JSON file named `history_baseline.json`:

```python
{
    "train_loss": [...],
    "train_accuracy": [...],
    "evaluation_loss": [...],
    "evaluation_accuracy": [...],
}
```

For accuracy, use `(torch.sigmoid(logits) >= 0.5)`. Use the number of examples,
not the number of batches, when averaging loss and accuracy.

## Question 3 — baseline accuracy

Calculate the median of `train_accuracy` in `history_baseline.json`. Submit the
value rounded to two decimal places. The grader accepts an absolute error of
`0.05`; the answer is not selected from a fixed option list.



In [4]:
import os
from torch.utils.data import Dataset
from PIL import Image

class HairDataset(Dataset):
    def __init__(self, data_dir, transform=None):
        self.data_dir = data_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []
        #self.classes = sorted(os.listdir(data_dir))
        # Include only non-hidden directories as classes
        self.classes = sorted([
            cls for cls in os.listdir(data_dir)
            if not cls.startswith(".")
            and os.path.isdir(os.path.join(data_dir, cls))
        ])
        self.class_to_idx = {cls: i for i, cls in enumerate(self.classes)}

        for label_name in self.classes:
            label_dir = os.path.join(data_dir, label_name)
            for img_name in os.listdir(label_dir):
                self.image_paths.append(os.path.join(label_dir, img_name))
                self.labels.append(self.class_to_idx[label_name])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, label

In [5]:
from torchvision import transforms

evaluation_transform = transforms.Compose([
    transforms.Resize((200, 200), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])



from torch.utils.data import DataLoader

train_dataset = HairDataset(
    data_dir='./data/train',
    transform=evaluation_transform
)

val_dataset = HairDataset(
    data_dir='./data/test',
    transform=evaluation_transform
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

   

In [6]:

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.002,
    momentum=0.8,
)
# Training loop
val_loss_history=[]
val_accuracy_history=[]
train_loss_history=[]
train_accuracy_history=[]
num_epochs = 10

for epoch in range(num_epochs):
    print(epoch)
    # Training phase
    model.train()  # Set the model to training mode
    running_loss = 0.0
    correct = 0
    total = 0

    # Iterate over the training data
    for inputs, labels in train_loader:
        # Move data to the specified device (GPU or CPU)
        inputs, labels = inputs.to(device), labels.to(device)

        # Zero the parameter gradients to prevent accumulation
        optimizer.zero_grad()
        # Forward pass
        outputs = model(inputs)
        # Calculate the loss
        labels = labels.float().view(-1, 1)
        loss = criterion(outputs, labels)
        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        # Accumulate training loss
        running_loss += loss.item()
        # Get predictions
        #_, predicted = torch.max(outputs.data, 1)
        predicted = (torch.sigmoid(outputs) >= 0.5).float()

        # Update total and correct predictions
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    # Calculate average training loss and accuracy
    train_loss = running_loss / len(train_loader)
    train_acc = correct / total
    train_loss_history.append(train_loss)
    train_accuracy_history.append(train_acc)
    # Validation phase
    model.eval()  # Set the model to evaluation mode
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    # Disable gradient calculation for validation
    with torch.no_grad():
        # Iterate over the validation data
        for inputs, labels in val_loader:
            # Move data to the specified device (GPU or CPU)
            inputs, labels = inputs.to(device), labels.to(device)
            labels = labels.float().view(-1, 1)
            # Forward pass
            outputs = model(inputs)
            predicted = (torch.sigmoid(outputs) >= 0.5).float()

            # Calculate the loss
            # Calculate the loss
            loss = criterion(outputs, labels)

            # Accumulate validation loss
            val_loss += loss.item()
            # Get predictions
            #_, predicted = torch.max(outputs.data, 1)




            
            # Update total and correct predictions
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    # Calculate average validation loss and accuracy
    val_loss /= len(val_loader)
    val_acc = val_correct / val_total
    val_loss_history.append(val_loss)
    val_accuracy_history.append(val_acc)
    # Print epoch results
    print(f'Epoch {epoch+1}/{num_epochs}')
    print(f'  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}')
    print(f'  Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}')

0
Epoch 1/10
  Train Loss: 0.6767, Train Acc: 0.5918
  Val Loss: 0.6862, Val Acc: 0.6269
1
Epoch 2/10
  Train Loss: 0.5586, Train Acc: 0.6816
  Val Loss: 0.6205, Val Acc: 0.6617
2
Epoch 3/10
  Train Loss: 0.5135, Train Acc: 0.7179
  Val Loss: 0.6492, Val Acc: 0.6766
3
Epoch 4/10
  Train Loss: 0.4978, Train Acc: 0.7516
  Val Loss: 0.6109, Val Acc: 0.6915
4
Epoch 5/10
  Train Loss: 0.6015, Train Acc: 0.7303
  Val Loss: 0.7823, Val Acc: 0.5871
5
Epoch 6/10
  Train Loss: 0.5842, Train Acc: 0.6767
  Val Loss: 0.6189, Val Acc: 0.6269
6
Epoch 7/10
  Train Loss: 0.5190, Train Acc: 0.7328
  Val Loss: 0.6084, Val Acc: 0.6567
7
Epoch 8/10
  Train Loss: 0.5001, Train Acc: 0.7503
  Val Loss: 0.6566, Val Acc: 0.6617
8
Epoch 9/10
  Train Loss: 0.4453, Train Acc: 0.8115
  Val Loss: 1.2908, Val Acc: 0.5672
9
Epoch 10/10
  Train Loss: 0.5542, Train Acc: 0.7491
  Val Loss: 0.6809, Val Acc: 0.6119


In [7]:
import json

history = {
    "train_loss": train_loss_history,
    "train_accuracy": train_accuracy_history,
    "evaluation_loss": val_loss_history,
    "evaluation_accuracy": val_accuracy_history,
}

with open("history_baseline.json", "w") as f:
    json.dump(history, f, indent=4)

print("Training history saved to history_baseline.json")


Training history saved to history_baseline.json


In [8]:
import json
import numpy as np
with open("history_baseline.json", "r") as f:
    history = json.load(f)

# Access the four series
train_accuracy = history["train_accuracy"]
round(float(np.median(np.array(train_accuracy))),2)

0.73

## Question 4 — baseline loss

Calculate the population standard deviation of `train_loss` in
`history_baseline.json` (for NumPy, use `np.std(values, ddof=0)`). Submit the
value rounded to three decimal places. The grader accepts an absolute error of
`0.02`.


In [9]:
import json

with open("history_baseline.json", "r") as f:
    history = json.load(f)

# Access the four series
train_loss = history["train_loss"]

round(float(np.std(train_loss, ddof=0)),3)


0.062

## Data augmentation

Keep the same model and optimizer. Replace only the training transform with:

```python
train_transform_augmented = transforms.Compose([
    transforms.Resize((200, 200), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.RandomRotation(
        50,
        interpolation=transforms.InterpolationMode.NEAREST,
    ),
    transforms.RandomResizedCrop(
        200,
        scale=(0.9, 1.0),
        ratio=(0.9, 1.1),
        interpolation=transforms.InterpolationMode.BILINEAR,
    ),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])
```

Do not recreate the model or optimizer. Train for 10 additional epochs and
write only those ten epochs to `history_augmented.json`. Continue using the
unchanged, deterministic `evaluation_dataset` and `evaluation_loader` for
every evaluation.

## Question 5 — augmented evaluation loss

Calculate the mean of `evaluation_loss` in `history_augmented.json`. Submit the
value rounded to three decimal places. The grader accepts an absolute error of
`0.02`.

In [10]:

from torchvision import transforms

train_transform_augmented = transforms.Compose([
    transforms.Resize((200, 200), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.RandomRotation(
        50,
        interpolation=transforms.InterpolationMode.NEAREST,
    ),
    transforms.RandomResizedCrop(
        200,
        scale=(0.9, 1.0),
        ratio=(0.9, 1.1),
        interpolation=transforms.InterpolationMode.BILINEAR,
    ),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])


from torch.utils.data import DataLoader

train_dataset = HairDataset(
    data_dir='./data/train',
    transform=train_transform_augmented
)

val_dataset = HairDataset(
    data_dir='./data/test',
    transform=train_transform_augmented
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

# Training loop
val_loss_history=[]
val_accuracy_history=[]
train_loss_history=[]
train_accuracy_history=[]
num_epochs = 10

for epoch in range(num_epochs):
    print(epoch)
    # Training phase
    model.train()  # Set the model to training mode
    running_loss = 0.0
    correct = 0
    total = 0

    # Iterate over the training data
    for inputs, labels in train_loader:
        # Move data to the specified device (GPU or CPU)
        inputs, labels = inputs.to(device), labels.to(device)

        # Zero the parameter gradients to prevent accumulation
        optimizer.zero_grad()
        # Forward pass
        outputs = model(inputs)
        # Calculate the loss
        labels = labels.float().view(-1, 1)
        loss = criterion(outputs, labels)
        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        # Accumulate training loss
        running_loss += loss.item()
        # Get predictions
        #_, predicted = torch.max(outputs.data, 1)
        predicted = (torch.sigmoid(outputs) >= 0.5).float()

        # Update total and correct predictions
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    # Calculate average training loss and accuracy
    train_loss = running_loss / len(train_loader)
    train_acc = correct / total
    train_loss_history.append(train_loss)
    train_accuracy_history.append(train_acc)
    # Validation phase
    model.eval()  # Set the model to evaluation mode
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    # Disable gradient calculation for validation
    with torch.no_grad():
        # Iterate over the validation data
        for inputs, labels in val_loader:
            # Move data to the specified device (GPU or CPU)
            inputs, labels = inputs.to(device), labels.to(device)
            labels = labels.float().view(-1, 1)
            # Forward pass
            outputs = model(inputs)
            predicted = (torch.sigmoid(outputs) >= 0.5).float()

            # Calculate the loss
            # Calculate the loss
            loss = criterion(outputs, labels)

            # Accumulate validation loss
            val_loss += loss.item()
            # Get predictions
            #_, predicted = torch.max(outputs.data, 1)




            
            # Update total and correct predictions
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    # Calculate average validation loss and accuracy
    val_loss /= len(val_loader)
    val_acc = val_correct / val_total
    val_loss_history.append(val_loss)
    val_accuracy_history.append(val_acc)
    # Print epoch results
    print(f'Epoch {epoch+1}/{num_epochs}')
    print(f'  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}')
    print(f'  Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}')

0
Epoch 1/10
  Train Loss: 0.7182, Train Acc: 0.5331
  Val Loss: 0.6625, Val Acc: 0.6318
1
Epoch 2/10
  Train Loss: 0.6788, Train Acc: 0.6642
  Val Loss: 0.6507, Val Acc: 0.6269
2
Epoch 3/10
  Train Loss: 0.6689, Train Acc: 0.5518
  Val Loss: 0.6513, Val Acc: 0.6269
3
Epoch 4/10
  Train Loss: 0.5814, Train Acc: 0.6667
  Val Loss: 0.6668, Val Acc: 0.5920
4
Epoch 5/10
  Train Loss: 0.5907, Train Acc: 0.6679
  Val Loss: 0.6825, Val Acc: 0.6667
5
Epoch 6/10
  Train Loss: 0.6121, Train Acc: 0.6704
  Val Loss: 0.6508, Val Acc: 0.6816
6
Epoch 7/10
  Train Loss: 0.6023, Train Acc: 0.6717
  Val Loss: 0.6481, Val Acc: 0.5970
7
Epoch 8/10
  Train Loss: 0.6087, Train Acc: 0.6729
  Val Loss: 0.5785, Val Acc: 0.6766
8
Epoch 9/10
  Train Loss: 0.5456, Train Acc: 0.6991
  Val Loss: 0.5872, Val Acc: 0.6915
9
Epoch 10/10
  Train Loss: 0.5762, Train Acc: 0.6991
  Val Loss: 0.6522, Val Acc: 0.5821


In [11]:
import json

history = {
    "train_loss": train_loss_history,
    "train_accuracy": train_accuracy_history,
    "evaluation_loss": val_loss_history,
    "evaluation_accuracy": val_accuracy_history,
}

with open("history_augmented.json", "w") as f:
    json.dump(history, f, indent=4)

print("Training history saved to history_augmented.json")


Training history saved to history_augmented.json


In [13]:
import json
import numpy as np

with open("history_augmented.json", "r") as f:
    history = json.load(f)

val_loss = history["evaluation_loss"]
mean_val_loss = round(float(np.mean(val_loss)),3)
mean_val_loss


0.643

## Question 6 — augmented evaluation accuracy

Calculate the mean of the last five values in `evaluation_accuracy` in
`history_augmented.json` (epochs 6 through 10 of the augmented phase). Submit
the value rounded to two decimal places. The grader accepts an absolute error
of `0.05`.

The tolerances cover minor CPU wheel differences, while the fixed seed, loader,
split, architecture, and inference transform prevent the large uncontrolled
variation from the 2025 exercise.

In [15]:
import json
import numpy as np

with open("history_augmented.json", "r") as f:
    history = json.load(f)

val_accuracy = history["evaluation_accuracy"]

mean_val_accuracy = round(float(np.mean(val_accuracy[-5:])),2)
print("Mean of the last five evaluation accuracies:",mean_val_accuracy)


Mean of the last five evaluation accuracies: 0.65
